In [1]:
# Cell 1 — Imports and connection
import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import psycopg2
conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()
print("Connected.")

Connected.


In [2]:
# Cell 2 — Schema introspection
cur.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'int'
    AND table_name = 'int_team_season_features'
    ORDER BY ordinal_position
""")
for row in cur.fetchall():
    print(row)

('team_name', 'text')
('season', 'integer')
('team_id', 'integer')
('conference', 'text')
('city', 'text')
('state', 'text')
('latitude', 'numeric')
('longitude', 'numeric')
('timezone', 'text')
('games_played', 'bigint')
('wins', 'bigint')
('losses', 'bigint')
('win_pct', 'numeric')
('avg_points_scored', 'numeric')
('avg_points_allowed', 'numeric')
('avg_point_diff', 'numeric')
('home_games', 'bigint')
('away_games', 'bigint')
('neutral_site_games', 'bigint')
('sp_rating', 'numeric')
('sp_ranking', 'integer')
('sp_offense', 'numeric')
('sp_defense', 'numeric')
('sp_special_teams', 'numeric')
('sp_offense_ranking', 'integer')
('sp_defense_ranking', 'integer')
('off_epa_per_play', 'numeric')
('off_epa_total', 'numeric')
('off_passing_epa', 'numeric')
('off_rushing_epa', 'numeric')
('def_epa_per_play', 'numeric')
('def_epa_total', 'numeric')
('def_passing_epa', 'numeric')
('def_rushing_epa', 'numeric')
('epa_differential', 'numeric')
('off_success_rate', 'numeric')
('off_pass_success_rat

In [3]:
# Cell 3 — Load candidate features, validate SP+ and recruiting columns
candidates = pd.read_csv(os.path.expanduser("~/cfb-analytics/artifacts/candidate_features.csv"))
candidates.columns = candidates.columns.str.strip().str.lower()
keep_cols = set(candidates.loc[candidates["keep"] == True, "column_name"].tolist())

sp_recruiting_features = [
    "sp_rating",
    "sp_offense",
    "sp_defense",
    "recruiting_3yr_avg",
]
print("Candidate feature validation:")
for f in sp_recruiting_features:
    status = "✓ keep" if f in keep_cols else "✗ NOT in keep list"
    print(f"  {f:40s} {status}")

Candidate feature validation:
  sp_rating                                ✓ keep
  sp_offense                               ✓ keep
  sp_defense                               ✓ keep
  recruiting_3yr_avg                       ✓ keep


In [6]:
# Cell 4 — Load season-level data
cur.execute("""
    SELECT
        s.team_name,
        s.season,
        s.sp_rating,
        s.sp_offense,
        s.sp_defense,
        s.off_epa_per_play,
        s.def_epa_per_play,
        s.avg_point_diff,
        s.avg_points_scored,
        s.avg_points_allowed,
        s.win_pct,
        s.recruiting_3yr_avg,
        c.conference
    FROM int.int_team_season_features s
    JOIN int.int_team_season_context c
        ON s.team_name = c.team_name AND s.season = c.season
    WHERE s.sp_rating IS NOT NULL
""")
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
sdf = pd.DataFrame(rows, columns=cols)

numeric_cols = [
    "sp_rating","sp_offense","sp_defense","off_epa_per_play","def_epa_per_play",
    "avg_point_diff","avg_points_scored","avg_points_allowed","win_pct","recruiting_3yr_avg"
]
sdf[numeric_cols] = sdf[numeric_cols].astype(float)

P4_CONFERENCES = {"ACC", "Big 12", "Big Ten", "SEC"}

def assign_tier(row):
    if row["team_name"] == "Notre Dame":
        return "P4"
    if row["team_name"] == "UConn":
        return "G5"
    if row["conference"] in P4_CONFERENCES:
        return "P4"
    return "G5"

sdf["tier"] = sdf.apply(assign_tier, axis=1)

print(f"Season-level rows      : {len(sdf):,}")
print(f"Seasons                : {sorted(sdf['season'].unique())}")
print(f"Tier split             : {sdf['tier'].value_counts().to_dict()}")
print(f"Conferences            : {sorted(sdf['conference'].unique())}")
print(f"\nNull counts:")
for col in numeric_cols:
    n = sdf[col].isna().sum()
    if n > 0:
        print(f"  {col:40s} {n:,}")

Season-level rows      : 534
Seasons                : [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Tier split             : {'G5': 288, 'P4': 246}
Conferences            : ['ACC', 'American Athletic', 'Big 12', 'Big Ten', 'Conference USA', 'FBS Independents', 'Mid-American', 'Mountain West', 'Pac-12', 'SEC', 'Sun Belt']

Null counts:
  recruiting_3yr_avg                       1


In [7]:
# Cell 5 — Load game-level data for game-level SP+ signal tests
cur.execute("""
    SELECT
        g.game_id,
        g.season,
        g.week,
        g.team_name,
        g.opponent,
        g.points_scored,
        g.points_allowed,
        g.close_game_epa_per_play,
        g.close_game_def_epa_per_play,
        g.opp_sp_rating_at_game_time,
        c.conference,
        c2.conference AS opp_conference,
        s.sp_rating          AS team_sp_rating,
        s.recruiting_3yr_avg AS team_recruiting
    FROM int.int_game_team_features g
    JOIN int.int_team_season_context c
        ON g.team_name = c.team_name AND g.season = c.season
    JOIN int.int_team_season_context c2
        ON g.opponent = c2.team_name AND g.season = c2.season
    JOIN int.int_team_season_features s
        ON g.team_name = s.team_name AND g.season = s.season
    WHERE g.points_scored IS NOT NULL
      AND g.points_allowed IS NOT NULL
      AND c.conference IN (
          'SEC','Big Ten','Big 12','ACC','Pac-12',
          'Mountain West','American Athletic','Sun Belt',
          'Mid-American','Conference USA','FBS Independents'
      )
""")
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
gdf = pd.DataFrame(rows, columns=cols)

game_numeric = [
    "points_scored","points_allowed","close_game_epa_per_play",
    "close_game_def_epa_per_play","opp_sp_rating_at_game_time",
    "team_sp_rating","team_recruiting"
]
gdf[game_numeric] = gdf[game_numeric].astype(float)
gdf["point_differential"] = gdf["points_scored"] - gdf["points_allowed"]
gdf["total_points"]       = gdf["points_scored"] + gdf["points_allowed"]
gdf["tier"] = gdf.apply(assign_tier, axis=1)

# True conference games only — opponent must be in same conference
# For independents, same_conference will be False — they have no conference opponents
# Notre Dame plays ACC schedule — their opp_conference will be ACC
# Include Notre Dame conference games by matching on opp_conference = 'ACC'
gdf_conf = gdf[gdf["conference"] == gdf["opp_conference"]].copy()

# Also include Notre Dame ACC schedule games
notre_dame_conf = gdf[
    (gdf["team_name"] == "Notre Dame") &
    (gdf["opp_conference"] == "ACC")
].copy()
notre_dame_conf["conference"] = "ACC"

gdf_conf = pd.concat([gdf_conf, notre_dame_conf], ignore_index=True)
gdf_conf = gdf_conf.drop_duplicates(subset=["game_id","team_name"])
gdf_conf = gdf_conf.sort_values(["team_name","season","week"])
gdf_conf["conf_game_num"] = gdf_conf.groupby(
    ["team_name","season"]
).cumcount() + 1

print(f"Game-level rows        : {len(gdf):,}")
print(f"True conference games  : {len(gdf_conf):,}")
print(f"Seasons                : {sorted(gdf['season'].unique())}")
print(f"Conferences in conf games: {sorted(gdf_conf['conference'].unique())}")
print(f"\nNull counts (game-level):")
for col in ["opp_sp_rating_at_game_time","team_sp_rating","team_recruiting",
            "close_game_epa_per_play","close_game_def_epa_per_play"]:
    n = gdf[col].isna().sum()
    pct = n / len(gdf) * 100
    print(f"  {col:40s} {n:,} ({pct:.1f}%)")

Game-level rows        : 5,996
True conference games  : 4,381
Seasons                : [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Conferences in conf games: ['ACC', 'American Athletic', 'Big 12', 'Big Ten', 'Conference USA', 'FBS Independents', 'Mid-American', 'Mountain West', 'Pac-12', 'SEC', 'Sun Belt']

Null counts (game-level):
  opp_sp_rating_at_game_time               1,523 (25.4%)
  team_sp_rating                           0 (0.0%)
  team_recruiting                          11 (0.2%)
  close_game_epa_per_play                  6 (0.1%)
  close_game_def_epa_per_play              6 (0.1%)


In [8]:
# Cell 5b — Fix conference game filter
# Remove FBS Independents from conf_game matching
# Notre Dame handled explicitly via ACC schedule

# Step 1: true conference games — both teams in same named conference, not independents
gdf_conf = gdf[
    (gdf["conference"] == gdf["opp_conference"]) &
    (gdf["conference"] != "FBS Independents")
].copy()

# Step 2: Notre Dame ACC schedule games
notre_dame_conf = gdf[
    (gdf["team_name"] == "Notre Dame") &
    (gdf["opp_conference"] == "ACC")
].copy()
notre_dame_conf = notre_dame_conf.copy()
notre_dame_conf["conference"] = "ACC"

gdf_conf = pd.concat([gdf_conf, notre_dame_conf], ignore_index=True)
gdf_conf = gdf_conf.drop_duplicates(subset=["game_id","team_name"])
gdf_conf = gdf_conf.sort_values(["team_name","season","week"])
gdf_conf["conf_game_num"] = gdf_conf.groupby(
    ["team_name","season"]
).cumcount() + 1

print(f"True conference games  : {len(gdf_conf):,}")
print(f"Conferences in conf games: {sorted(gdf_conf['conference'].unique())}")
print(f"\nConference game number distribution:")
print(gdf_conf["conf_game_num"].value_counts().sort_index().head(15))
print(f"\nNull rate opp_sp_rating_at_game_time in conf games:")
n_null = gdf_conf["opp_sp_rating_at_game_time"].isna().sum()
print(f"  {n_null:,} / {len(gdf_conf):,} ({n_null/len(gdf_conf)*100:.1f}%)")
print(f"  By season:")
for s, grp in gdf_conf.groupby("season"):
    n = grp["opp_sp_rating_at_game_time"].isna().sum()
    print(f"    {s}: {n:,} / {len(grp):,} null ({n/len(grp)*100:.1f}%)")

True conference games  : 4,357
Conferences in conf games: ['ACC', 'American Athletic', 'Big 12', 'Big Ten', 'Conference USA', 'Mid-American', 'Mountain West', 'Pac-12', 'SEC', 'Sun Belt']

Conference game number distribution:
conf_game_num
1     522
2     520
3     518
4     518
5     517
6     516
7     514
8     502
9     202
10     28
Name: count, dtype: int64

Null rate opp_sp_rating_at_game_time in conf games:
  1,090 / 4,357 (25.0%)
  By season:
    2022: 1,050 / 1,050 null (100.0%)
    2023: 16 / 1,098 null (1.5%)
    2024: 8 / 1,085 null (0.7%)
    2025: 16 / 1,124 null (1.4%)
